# Three useful candidate features

This notebook runs by itself. It uses visible products from one session and gives each product a few simple facts.

A model cannot understand a product ID by itself. We give it simple facts about each candidate product.

In [ ]:
from pathlib import Path
import sys

import polars as pl

start_path = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in [start_path, *start_path.parents] if (path / "src").is_dir())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.session_features import add_session_features
from src.training.validation import split_observed_hidden

events = pl.read_parquet(PROJECT_ROOT / "data/processed/train_dev.parquet")
session_id = events.select("session").unique().sort("session").item(0, 0)
session_events = events.filter(pl.col("session") == session_id)
observed, _ = split_observed_hidden(session_events)

candidates = observed.select("session", pl.col("aid").alias("candidate")).unique()
featured_candidates = add_session_features(candidates, observed)

featured_candidates.select(
    "candidate",
    "event_count",
    "last_position",
    "was_clicked",
    "was_carted",
    "was_ordered",
).sort("last_position", descending=True).head(20)

Read the important columns this way:

- `event_count`: how many times this product appeared in the visible session.
- `last_position`: where the candidate last appeared in the visible session. Positions count from oldest to newest, so a larger value means it was seen more recently.
- `was_clicked`, `was_carted`, and `was_ordered`: whether the visible session already contained that kind of action for this product.

Later, the same facts will be added to newly retrieved products too. For a product that was not already in the session, its event count is `0`.